
# 1. Environment Setup and Initialization
Import common libraries and mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import re
import time
import pandas as pd
from pathlib import Path

## 2. Sentence Splitter
This section normalizes line breaks, preserves specific labels (e.g., "Quality:", "Effectiveness:"), and splits long reviews into individual sentences. The parsed sentences are then structured and saved into CSV and JSON formats.

In [ ]:
INPUT_FOLDER = Path("/content/drive/MyDrive/Colab Notebooks/TNL/Project/DATASET")

OUTPUT_FOLDER = Path("/content/drive/MyDrive/Colab Notebooks/TNL/Project/PREPROCESS DATASET")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_FOLDER / "01_raw_sentences.csv"
OUTPUT_JSON = OUTPUT_FOLDER / "01_raw_sentences.json"

# =====================================================
# SENTENCE SPLITTER
# =====================================================
def split_review(text):
    if pd.isna(text):
        return []
    text = str(text).strip()
    if text == "":
        return []

    # Normalize line breaks
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)

    # Preserve labels (e.g. "Quality: Good")
    text = re.sub(r'([A-Za-z]+)\s*:\s*', r'\1: ', text)

    # New line -> sentence separator
    text = text.replace("\n", ". ")

    # Split sentence based on punctuation
    sentences = re.split(r'(?<=[.!?])\s*', text)
    cleaned = []

    for sentence in sentences:
        sentence = sentence.strip()
        sentence = re.sub(r"\s+", " ", sentence)
        sentence = sentence.rstrip(".!? ")

        if sentence == "":
            continue

        # Ignore punctuation only sentences
        if re.fullmatch(r"[-_~.,!?]+", sentence):
            continue

        cleaned.append(sentence)

    return cleaned

# =====================================================
# PROCESS DATASET
# =====================================================
all_rows = []
sentence_id = 1
json_files = sorted(INPUT_FOLDER.glob("*.json"))

print("=" * 60)
print(f"Found {len(json_files)} JSON files")
print("=" * 60)

for json_file in json_files:
    print(f"Processing: {json_file.name}")
    product_id = json_file.stem

    with open(json_file, "r", encoding="utf-8") as f:
        reviews = json.load(f)

    for review in reviews:
        comment = str(review.get("comment", "")).strip()
        if comment == "":
            continue

        sentences = split_review(comment)

        for sentence in sentences:
            all_rows.append({
                "sentence_id": sentence_id,
                "source_file": json_file.name,
                "product_id": product_id,
                "category": review.get("category", ""),
                "brand": review.get("brand", ""),
                "product_name": review.get("product_name", ""),
                "rating": review.get("rating", None),
                "review_date": review.get("review_date", ""),
                "original_sentence": sentence,
                "translated_sentence": None,
                "emoji": None,
                "emoji_sentiment": None,
                "aspect_keywords": [],
                "clean_sentence": None,
                "relevant": None,
                "text_sentiment": None,
                "final_sentiment": None
            })
            sentence_id += 1

# =====================================================
# SAVE FILES
# =====================================================
df = pd.DataFrame(all_rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("SENTENCE SPLIT COMPLETED")
print("=" * 60)
print(f"Products  : {len(json_files)}")
print(f"Sentences : {sentence_id - 1:,}")

print("\nSaved Files")
print(OUTPUT_CSV)
print(OUTPUT_JSON)

### Preview Split Sentences
Let's verify the sentence splitting logic by previewing the first few sentences from each product in the dataset.

In [ ]:
# =====================================================
# PREVIEW DATA
# =====================================================
preview_count = 5
products_seen = {}

for row in all_rows:
    product = row["product_id"]

    if product not in products_seen:
        products_seen[product] = 0
        print("\n" + "=" * 70)
        print(product)
        print("=" * 70)

    if products_seen[product] < preview_count:
        print(f"Sentence ID : {row['sentence_id']}")
        print(f"Original    : {row['original_sentence']}")
        products_seen[product] += 1

## 3. Text Normalization and Translation
This module handles the installation of required translation libraries, normalizes the text based on a predefined glossary/dictionary, and utilizes `deep-translator` to translate non-English sentences into English.

In [ ]:
# =====================================================
# INSTALL REQUIRED LIBRARIES
# =====================================================
!pip install deep-translator
!pip install contractions

In [ ]:
import time
import contractions
from deep_translator import GoogleTranslator

# =====================================================
# CONFIG & LOAD NORMALIZATION
# =====================================================
INPUT_FILE = OUTPUT_FOLDER / "01_raw_sentences.json"
OUTPUT_FILE = OUTPUT_FOLDER / "02_sentence_translated.json"
NORMALIZATION_FILE = Path("/content/drive/MyDrive/Colab Notebooks/TNL/Project/CONFIG/normalization.json")

with open(NORMALIZATION_FILE, "r", encoding="utf-8") as f:
    NORMALIZATION = json.load(f)

ALL_NORMALIZE = {}
for group in NORMALIZATION.values():
    ALL_NORMALIZE.update(group)

print(f"Loaded {len(ALL_NORMALIZE)} normalization rules.")

# =====================================================
# HELPER FUNCTIONS
# =====================================================
def normalize_sentence(sentence):
    sentence = str(sentence)

    # Dictionary Normalization
    for wrong, correct in ALL_NORMALIZE.items():
        sentence = re.sub(rf"\b{re.escape(wrong)}\b", correct, sentence, flags=re.IGNORECASE)

    # English contractions
    sentence = contractions.fix(sentence)

    # Remove extra spaces
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

# Initialize Google Translator
translator = GoogleTranslator(source="auto", target="en")

def translate_sentence(sentence):
    sentence = str(sentence).strip()
    if not sentence:
        return sentence

    try:
        translated = translator.translate(sentence)
        return translated.strip() if translated else sentence
    except Exception as e:
        print(f"Translation Error: {e}")
        return sentence

# =====================================================
# EXECUTE NORMALIZATION & TRANSLATION
# =====================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

total = len(data)
normalized_count = 0
translation_cache = {}
translated_count = 0
unchanged_count = 0
cache_hits = 0

print(f"Loaded {total:,} sentences for translation.\n")

for i, row in enumerate(data):
    # Step 1: Normalize
    original = row["original_sentence"]
    normalized = normalize_sentence(original)
    row["normalized_sentence"] = normalized

    if normalized != original:
        normalized_count += 1

    # Step 2: Translate
    if normalized in translation_cache:
        translated = translation_cache[normalized]
        cache_hits += 1
    else:
        translated = translate_sentence(normalized)
        translation_cache[normalized] = translated

    row["translated_sentence"] = translated

    if translated == original:
        unchanged_count += 1
    else:
        translated_count += 1

    # Progress tracking
    if (i + 1) % 200 == 0 or (i + 1) == total:
        print(f"Processed {i+1}/{total}")
        time.sleep(0.3)

# Save Translated Data
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# =====================================================
# TRANSLATION SUMMARY
# =====================================================
print("\n" + "="*60)
print("TRANSLATION SUMMARY")
print("="*60)
print(f"Total Sentences : {total:,}")
print(f"Normalized      : {normalized_count:,}")
print(f"Translated      : {translated_count:,}")
print(f"Unchanged       : {unchanged_count:,}")
print(f"Cache Hits      : {cache_hits:,}")
print(f"Unique Requests : {len(translation_cache):,}")
print(f"\nSaved -> {OUTPUT_FILE}")

Translation Validation

In [ ]:
INPUT_FILE = OUTPUT_FOLDER / "02_sentence_translated.json"
OUTPUT_FILE = OUTPUT_FOLDER / "03_need.manual_translation_review.csv"
GLOSSARY_FILE = Path(r"/content/drive/MyDrive/Colab Notebooks/TNL/Project/CONFIG/translation_glossary.json")

In [ ]:
with open(INPUT_FILE,"r",encoding="utf-8") as f:
    data=json.load(f)

with open(GLOSSARY_FILE,"r",encoding="utf-8") as f:
    glossary=json.load(f)

malay_words=set(glossary["malay"])
bad_translation=glossary["bad_translation"]

In [ ]:
def tokenize(text):
    return re.findall(r"[A-Za-z']+", str(text).lower())

def contains_chinese(text):
    return bool(re.search( r'[\u4e00-\u9fff]', str(text)))

def contains_malay(text):
    tokens=tokenize(text)
    remain=[]
    for t in tokens:
        if t in malay_words:
            remain.append(t)
    return remain


def original_has_non_english(text):
    if contains_chinese(text):
        return True
    if contains_malay(text):
        return True
    return False

In [ ]:
manual_review=[]
for row in data:
    original=row["original_sentence"]
    translated=row["translated_sentence"]
    reason=[]
    if original_has_non_english(original):
        if original.lower()==translated.lower():
            reason.append("Not Translated")

    remain=contains_malay(translated)
    if remain:
        reason.append("Remaining Malay: "+", ".join(sorted(set(remain))))

    if contains_chinese(translated):
        reason.append(
            "Remaining Chinese"
        )

    text=translated.lower()
    for bad in bad_translation:
        if bad in text:
            reason.append("Suspicious: "+bad)

    if reason:
        row["review_reason"]=" | ".join(reason)
        manual_review.append(row)

In [ ]:
review_df = pd.DataFrame(manual_review)
print("=" * 60)
print("Translation Validation Report")
print("=" * 60)
print(f"Total Sentences : {len(data)}")
print(f"Need Review     : {len(review_df)}")
print("=" * 60)
display(
    review_df[
        ["sentence_id", "original_sentence", "translated_sentence",  "review_reason"]].head(50))

review_df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")

print("\nSaved:")
print(OUTPUT_FILE)

In [ ]:
NEED_MANUAL_UPDATED_JSON = OUTPUT_FOLDER / "03_need.manual_translation_review.csv"
manual_json = []
for row in manual_review:
    manual_json.append({
        "sentence_id": row["sentence_id"],
        "original_sentence": row["original_sentence"],
        "google_translation": row["translated_sentence"],
        # Fill this manually later
        "manual_translation": "",
        # Final translation (will be generated later)
        "final_translation": row["translated_sentence"],
        # pending / completed
        "status": "pending",
        "review_reason": row["review_reason"]
    })

with open(NEED_MANUAL_UPDATED_JSON, "w", encoding="utf-8") as f:
    json.dump(manual_json, f, ensure_ascii=False, indent=2)

print("\nManual Translation Review JSON Created!")
print(NEED_MANUAL_UPDATED_JSON)
print(f"Total Review Records : {len(manual_json)}")

In [ ]:
#Merge Script
# =====================================================
# CONFIG
TRANSLATED_FILE = OUTPUT_FOLDER / "02_sentence_translated.json"
MANUAL_FILE = Path("/content/drive/MyDrive/Colab Notebooks/TNL/Project/CONFIG/updated_manual_translation_review.json")
OUTPUT_FILE = OUTPUT_FOLDER / "04_sentence_translated_completed.json"

# =====================================================
# LOAD FILES
with open(TRANSLATED_FILE, "r", encoding="utf-8"
) as f: translated_data = json.load(f)

with open( MANUAL_FILE, "r", encoding="utf-8"
) as f: manual_data = json.load(f)

# =====================================================
# BUILD LOOKUP
manual_lookup = {}

for row in manual_data:
    manual_lookup[row["sentence_id"]] = row

# =====================================================
# MERGE
corrected = 0

for row in translated_data:
    sid = row["sentence_id"]
    if sid not in manual_lookup:
        row["translation_source"] = "google"
        continue

    manual = manual_lookup[sid]

    if manual["status"] == "completed":
        row["google_translation"] = row["translated_sentence"]
        row["translated_sentence"] = manual["final_translation"]
        row["translation_source"] = "manual"
        corrected += 1
    else:
        row["translation_source"] = "google"

# =====================================================
# SAVE
with open(OUTPUT_FILE, "w", encoding="utf-8"
) as f:
    json.dump(translated_data, f, ensure_ascii=False, indent=2)

# =====================================================
# SUMMARY
print("=" * 60)
print("MERGE COMPLETED")
print("=" * 60)

print(f"Total Sentences : {len(translated_data):,}")
print(f"Manual Applied  : {corrected:,}")
print(f"Google Retained : {len(translated_data)-corrected:,}")

print()
print("Saved to:")
print(OUTPUT_FILE)

## 4. Emoji Detection & Sentiment Mapping
In this section, we extract emojis from the translated sentences. We use a custom emoji dictionary alongside an official emoji sentiment ranking dataset to assign positive, neutral, or negative sentiment scores to each sentence based on the emojis it contains.

In [ ]:
# =====================================================
# INSTALL & IMPORT
# =====================================================
!pip install emoji
import emoji
from collections import Counter

# =====================================================
# CONFIG & LOAD EMOJI DICTIONARIES
# =====================================================
INPUT_FILE = OUTPUT_FOLDER / "04_sentence_translated_completed.json"
OUTPUT_FILE = OUTPUT_FOLDER / "05_emoji_scored.json"
DICT_FILE = "/content/drive/MyDrive/Colab Notebooks/TNL/Project/EMOJI DICTIONARIES/emoji_dictionary.json"
EMOJI_FILE = "/content/drive/MyDrive/Colab Notebooks/TNL/Project/EMOJI DICTIONARIES/Emoji_Sentiment_Data_v1.0.csv" # https://kt.ijs.si/data/Emoji_sentiment_ranking/

# Load Custom Emoji Dictionary
with open(DICT_FILE, "r", encoding="utf-8") as f:
    EMOJI_DICT = json.load(f)

# Load Emoji Sentiment Ranking
emoji_df = pd.read_csv(EMOJI_FILE)
EMOJI_RANKING = {}

OVERRIDE_SCORE = {
    "🚚": 0.0, "📦": 0.0, "🛒": 0.0, "💰": 0.0,
    "🏃": 0.0, "🚴": 0.0, "🍫": 0.0, "⚡": 0.0
}

for _, row in emoji_df.iterrows():
    total = row["Positive"] + row["Neutral"] + row["Negative"]
    if total == 0:
        continue

    if row["Emoji"] in OVERRIDE_SCORE:
        score = OVERRIDE_SCORE[row["Emoji"]]
    else:
        score = (row["Positive"] - row["Negative"]) / total

    EMOJI_RANKING[row["Emoji"]] = {
        "meaning": row["Unicode name"],
        "category": row["Unicode block"],
        "score": score,
        "positive": row["Positive"] / total,
        "neutral": row["Neutral"] / total,
        "negative": row["Negative"] / total
    }

# Merge Ranking into Custom Dictionary
for e, info in EMOJI_RANKING.items():
    if e in EMOJI_DICT:
        # Keep original custom meaning/category
        EMOJI_DICT[e]["score"] = info["score"]
        EMOJI_DICT[e]["positive"] = info["positive"]
        EMOJI_DICT[e]["neutral"] = info["neutral"]
        EMOJI_DICT[e]["negative"] = info["negative"]
    else:
        # Present in official ranking, missing in custom dict
        EMOJI_DICT[e] = info

# Add Custom New Emojis
NEW_EMOJI = {
    "👐": {"meaning": "open hands", "category": "People & Body", "score": 0.30}
}
for e, info in NEW_EMOJI.items():
    if e not in EMOJI_DICT:
        EMOJI_DICT[e] = info

# Save Merged Dictionary
with open(DICT_FILE, "w", encoding="utf-8") as f:
    json.dump(EMOJI_DICT, f, ensure_ascii=False, indent=4)

print(f"Merged Dictionary Saved! Total Emojis: {len(EMOJI_DICT)}")

In [ ]:
# =====================================================
# EMOJI EXTRACTION AND SCORING
# =====================================================
SKIN_TONES = {"🏻", "🏼", "🏽", "🏾", "🏿"}
GENDER_MAP = {"🤷♀": "🤷", "🤷♂": "🤷", "🏃♀": "🏃", "🏃♂": "🏃", "🚴♀": "🚴", "🚴♂": "🚴"}

def normalize_emoji(e):
    e = e.replace("\ufe0f", "") # Remove Variation Selector
    e = e.replace("\u200d", "") # Remove Zero Width Joiner
    for tone in SKIN_TONES:
        e = e.replace(tone, "") # Remove Skin Tone
    if e in GENDER_MAP:
        e = GENDER_MAP[e] # Normalize Gender Emoji
    return e

unknown_emojis = set()

data = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

for row in data:
    sentence = str(row.get("translated_sentence", ""))
    emoji_objects = emoji.emoji_list(sentence)

    emoji_list, emoji_unique = [], []
    emoji_meaning, emoji_summary = [], []
    emoji_category, emoji_category_unique = [], []
    emoji_scores = []

    for obj in emoji_objects:
        e = normalize_emoji(obj["emoji"])
        emoji_list.append(e)
        if e not in emoji_unique:
            emoji_unique.append(e)

        info = EMOJI_DICT.get(e)

        if info is None:
            emoji_meaning.append("unknown")
            emoji_category.append("unknown")
            if "unknown" not in emoji_summary: emoji_summary.append("unknown")
            if "unknown" not in emoji_category_unique: emoji_category_unique.append("unknown")
            unknown_emojis.add(e)
        else:
            emoji_meaning.append(info["meaning"])
            emoji_category.append(info["category"])
            if info["meaning"] not in emoji_summary: emoji_summary.append(info["meaning"])
            if info["category"] not in emoji_category_unique: emoji_category_unique.append(info["category"])
            emoji_scores.append(info.get("score", 0))

    # Calculate Sentiment and Intensity
    emoji_score = sum(emoji_scores) / len(emoji_scores) if emoji_scores else 0

    emoji_sentiment = "positive" if emoji_score >= 0.20 else "negative" if emoji_score <= -0.20 else "neutral"

    if abs(emoji_score) >= 0.60: emoji_intensity = "strong"
    elif abs(emoji_score) >= 0.30: emoji_intensity = "medium"
    elif abs(emoji_score) >= 0.05: emoji_intensity = "weak"
    else: emoji_intensity = "none"

    # Save to row
    row.update({
        "emoji": emoji_list,
        "emoji_unique": emoji_unique,
        "emoji_count": len(emoji_list),
        "emoji_unique_count": len(emoji_unique),
        "emoji_meaning": emoji_meaning,
        "emoji_summary": emoji_summary,
        "emoji_category": emoji_category,
        "emoji_category_unique": emoji_category_unique,
        "emoji_score": emoji_score,
        "emoji_sentiment": emoji_sentiment,
        "emoji_intensity": emoji_intensity
    })

# Save output
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("\nEmoji Extraction Completed & Saved!")
print(f"Total Sentences: {len(data)}")
print(f"Unknown Emojis found: {len(unknown_emojis)}")

In [ ]:
# UNKNOWN EMOJI REPORT
print("=" * 60)
print("Unknown Emoji")
print("=" * 60)

unknown_list = sorted(unknown_emojis)

print(unknown_list)
print(f"Total Unknown : {len(unknown_list)}")

### Emoji Statistics & Preview
Let's generate a quick summary of the emojis found in our dataset and preview the extracted data.

In [ ]:
# =====================================================
# EMOJI STATISTICS
# =====================================================
emoji_counter = Counter()
meaning_counter = Counter()
category_counter = Counter()
sentiment_counter = Counter()
intensity_counter = Counter()
contain_emoji = 0

for row in data:
    if row["emoji_count"] > 0:
        contain_emoji += 1
    emoji_counter.update(row["emoji"])

    meaning_counter.update(row["emoji_summary"])
    category_counter.update(row["emoji_category_unique"])
    sentiment_counter.update([row["emoji_sentiment"]])
    intensity_counter.update( [row["emoji_intensity"]])

print("=" * 60)
print("Emoji Statistics")
print("=" * 60)
print(f"Total Sentences : {len(data)}")
print(f"Sentences containing Emojis : {contain_emoji}")
print(f"Sentences without Emojis    : {len(data) - contain_emoji}")


In [ ]:
print("=" * 60)
print("Top 15 Emoji")
print("=" * 60)
for e, c in emoji_counter.most_common(15):
    print(f"{e:<5} {c}")


In [ ]:
print("=" * 60)
print("Top Emoji Meanings")
print("=" * 60)
for m, c in meaning_counter.most_common():
    print(f"{m:<20} {c}")

In [ ]:
print("=" * 60)
print("Emoji Categories")
print("=" * 60)
for m, c in category_counter.items():
    print(f"{m:<15} {c}")

In [ ]:
print("=" * 60)
print("Emoji Sentiment")
print("=" * 60)

for m, c in sentiment_counter.items():
    print(f"{m:<15} {c}")

In [ ]:
# Preview Data
df_emoji = pd.DataFrame(data)
display(
    df_emoji.query("emoji_count > 0")[[
        "translated_sentence", "emoji", "emoji_unique",
        "emoji_count", "emoji_unique_count", "emoji_meaning",
        "emoji_summary", "emoji_category", "emoji_category_unique",
        "emoji_score",  "emoji_sentiment", "emoji_intensity"
    ]].head(10)
)

## 5. Text Cleaning (Regex) & Relevance Filtering
This module performs deep text cleaning by stripping URLs, HTML tags, control characters, and normalizing unicode formats. After cleaning, we filter out irrelevant reviews (e.g., spam, greeting-only, "review for coins" statements) so that only meaningful product reviews remain.

In [ ]:
# =====================================================
# TEXT CLEANING (REGEX)
# =====================================================
import unicodedata

INPUT_FILE = OUTPUT_FOLDER / "05_sentence_emoji.json"
OUTPUT_CLEAN_FILE = OUTPUT_FOLDER / "06_sentence_clean.json"

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
HTML_PATTERN = re.compile(r"<.*?>")
MULTI_SPACE = re.compile(r"\s+")
REPEAT_PUNCT = re.compile(r"([!?.,])\1+")
CONTROL_CHAR = re.compile(r"[\r\n\t]")
REPEAT_LETTER = re.compile(r"(.)\1{2,}")
MULTI_HYPHEN = re.compile(r"-{2,}")

def clean_text(text):
    if text is None: return ""
    text = str(text)

    # Unicode Normalization & Fancy Font fix
    text = unicodedata.normalize("NFKC", text)
    table = str.maketrans({"🅰":"A", "🅱":"B", "🅲":"C", "🅳":"D", "🅴":"E", "🅵":"F",
        "🅶":"G", "🅷":"H", "🅸":"I", "🅹":"J", "🅺":"K", "🅻":"L", "🅼":"M", "🅽":"N", "🅾":"O",
        "🅿":"P", "🆀":"Q", "🆁":"R", "🆂":"S", "🆃":"T", "🆄":"U", "🆅":"V",  "🆆":"W", "🆇":"X",
        "🆈":"Y", "🆉":"Z"}) # Truncated mapping for brevity
    text = text.translate(table)

    text = URL_PATTERN.sub(" ", text)
    text = HTML_PATTERN.sub(" ", text)
    text = emoji.replace_emoji(text, replace="") # Remove emojis from text string
    text = CONTROL_CHAR.sub(" ", text)
    text = text.lower()
    text = REPEAT_PUNCT.sub(r"\1", text)
    text = REPEAT_LETTER.sub(r"\1\1", text)
    text = MULTI_HYPHEN.sub("-", text)
    text = re.sub(r"[^a-z0-9\s%$:/.,()&+\-']", " ", text) # Keep only specific chars
    text = MULTI_SPACE.sub(" ", text)

    return text.strip()

def refine_sentence(text):
    text = str(text)
    text = re.sub(r"\.+$", "", text) # Remove ending period
    text = re.sub(r"\.\s*([A-Za-z])", lambda m: ", " + m.group(1).lower(), text) # Middle period to comma
    text = re.sub(r",\s*,+", ", ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply Cleaning
modified_count = 0
empty_count = 0
for row in data:
    original = str(row.get("translated_sentence", ""))
    cleaned = refine_sentence(clean_text(original))
    row["clean_sentence"] = cleaned
    if cleaned != original:
        modified_count += 1

print(f"Total Sentences : {len(data):,}")
print(f"Modified        : {modified_count:,}")
print(f"Empty           : {empty_count:,}")

In [ ]:
# SAMPLE RESULT
sample_df = pd.DataFrame(data)

display(
    sample_df.sample(20, random_state=42)[
        ["translated_sentence", "clean_sentence"]
    ])

# SAVE
OUTPUT_FILE = OUTPUT_FOLDER / "06_sentence_clean.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Saved")
print(OUTPUT_FILE)

In [ ]:
# =====================================================
# RELEVANCE FILTERING & ASSIGNING STATUS
# =====================================================
OUTPUT_FILE_07 = OUTPUT_FOLDER / "07_relevance_full_tagged.json"
OUTPUT_FILE_08 = OUTPUT_FOLDER / "08_relevant_filtered.json"

GREETING = {"thank you", "thanks", "thanks seller", "thank you seller", "tq", "tq seller", "tks", "ok thanks", "thank u",
    "thanks alot", "thanks a lot", "many thanks", "thx", "thx seller", "ty", "ty seller", "appreciate it"}
COIN_REVIEW = {"earning coins", "receive coins", "review for coins", "collect coins"}
SPAM_WORDS = {"asdf", "abcdef","qwerty","asdfgh", "xxxxx"}
PRODUCT_ATTRIBUTES = ("quality:", "effectiveness:", "taste:", "delivery:", "packaging:", "price:", "expiry:","seller:")
NON_SENTIMENT_EMOJI = {"delivery", "running", "cycling", "package"}

def check_relevance(row):
    sentence = str(row.get("clean_sentence", "")).strip().lower()
    has_emo = row.get("emoji_count", 0) > 0
    categories = row.get("emoji_category_unique", [])
    has_sentiment_emo = "positive" in categories or "negative" in categories

    if sentence == "":
        if has_emo: return ("emoji_review", "Emoji Review") if has_sentiment_emo else ("emoji_only", "Neutral Emoji Only")
        return "empty", "Empty after Cleaning"

    for keyword in COIN_REVIEW:
        if keyword in sentence: return "coin_review", "Coin Review"

    if sentence in GREETING:
        return ("emoji_review", "Greeting + Sentiment Emoji") if has_sentiment_emo else ("greeting", "Greeting Only")

    if sentence in SPAM_WORDS or re.fullmatch(r"(.)\1{5,}", sentence):
        return "spam", "Meaningless Text"

    if bool(re.fullmatch(r"[0-9\s:./-]+", sentence)):
        return "numeric_only", "Numeric Only"

    if sentence.startswith(PRODUCT_ATTRIBUTES):
        value = sentence.split(":", 1)[1].strip()
        if value != "": return "relevant", "Product Attribute"
        if has_emo: return "relevant", "Product Attribute + Emoji"
        return "incomplete", "Incomplete Product Attribute"

    words = sentence.split()
    if len(words) <= 1 and has_sentiment_emo:
        return "emoji_review", "Emoji Review"

    return "relevant", "General Product Review"

# Apply Relevance Logic
from collections import Counter
status_counter = Counter()
relevant_data = []

for row in data:
    status, reason = check_relevance(row)
    row["relevance_status"] = status
    row["relevance_reason"] = reason
    status_counter[status] += 1

    if status in ["relevant", "emoji_review"]:
        relevant_data.append(row)

# 1. Save the FULL dataset with all tags
with open(OUTPUT_FILE_07, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# 2. Save ONLY the filtered/relevant dataset
with open(OUTPUT_FILE_08, "w", encoding="utf-8") as f:
    json.dump(relevant_data, f, ensure_ascii=False, indent=2)

print("\nRelevance Filtering Completed!")
for status, count in sorted(status_counter.items()):
    print(f"{status:<15} : {count}")

print(f"\nOriginal Dataset : {len(data):,} saved to 07_sentence_relevance.json")
print(f"Relevant Dataset : {len(relevant_data):,} saved to 08_sentence_relevant.json")

In [ ]:
# =====================================================
# MANUAL REVIEW
# =====================================================
review_df = df[df["relevance_status"] != "relevant"][[
    "sentence_id", "translated_sentence", "clean_sentence",
    "emoji_summary", "emoji_score", "relevance_status",
    "relevance_reason"
]]

print("=" * 60)
print(f"Need Manual Review : {len(review_df)}")
print("=" * 60)

display(review_df)

In [ ]:
# =====================================================
# RELEVANT DATASET EXTRACTION
# =====================================================
relevant_data = [row
    for row in data
    if row["relevance_status"] in [ "relevant", "emoji_review"]
]

print("=" * 60)
print("Relevant Dataset Extraction")
print("=" * 60)

print(f"Original Dataset : {len(data):,}")
print(f"Relevant Dataset : {len(relevant_data):,}")
print(f"Removed          : {len(data)-len(relevant_data):,}")

## 6. Stop Word Removal
To reduce noise in our dataset, we remove common English stopwords (like "the", "is", "at"). However, we carefully preserve sentiment-bearing words (like "not", "very", "recommend") and domain-specific keywords (like "quality", "taste") to ensure we don't lose the core meaning of the reviews.

In [ ]:
# =====================================================
# INSTALL & IMPORT
# =====================================================
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

# =====================================================
# CONFIG & BUILD CUSTOM STOPWORD LIST
# =====================================================
INPUT_FILE = OUTPUT_FOLDER / "08_relevant_filtered.json"
OUTPUT_FILE = OUTPUT_FOLDER / "09_stopwords_removed.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 60)
print(f"Loaded {len(data):,} relevant reviews for stopword removal")
print("=" * 60)

# Load base NLTK stopwords
STOPWORDS = set(stopwords.words("english"))

# Keep Important Sentiment Words
KEEP_WORDS = {
    # Negation
    "no", "nor", "not", "never",
    "don't", "doesn't", "didn't", "can't", "couldn't", "won't",
    "wouldn't", "shouldn't", "haven't", "hasn't", "hadn't",
    "isn't", "aren't", "wasn't", "weren't",
    # Degree
    "very", "too", "more", "most", "less", "least", "quite", "really",
    # Review words
    "again", "recommend", "recommended",
    # Aspect Words
    "quality", "effectiveness", "taste", "delivery", "packaging",
    "price", "seller", "service", "expiry", "product", "products",
    "flavour", "flavours", "flavor", "flavors"
}

STOPWORDS = STOPWORDS - KEEP_WORDS

# Additional Custom Stopwords
CUSTOM_STOPWORDS = {
    "whether", "inside", "bit", "so", "towards", "onto", "upon",
    "across", "among", "amongst", "beside", "besides", "around",
    "within", "throughout", "through", "via", "per"
}

STOPWORDS |= CUSTOM_STOPWORDS

print(f"Base Stopwords : {len(STOPWORDS)}")
print(f"Protected Words: {len(KEEP_WORDS)}")
print(f"Custom Added   : {len(CUSTOM_STOPWORDS)}")

In [ ]:
# =====================================================
# REMOVE STOPWORDS FUNCTION
# =====================================================
def remove_duplicate_tokens(sentence):
    tokens = sentence.split()
    if not tokens:
        return sentence

    result = [tokens[0]]
    for token in tokens[1:]:
        if token != result[-1]:
            result.append(token)
    return " ".join(result)

# Apply Removal
for row in data:
    sentence = row.get("clean_sentence", "")

    # Advanced Tokenization
    tokens = re.findall(
      r"\d+(?:st|nd|rd|th)|\d+/\d+|\d+%|[A-Za-z]+(?:'[A-Za-z]+)?|[^\w\s]",
      sentence
    )

    filtered_tokens = []
    for token in tokens:
        token_lower = token.lower()

        # Keep punctuation
        if re.fullmatch(r"[,.;:!?&()-]", token):
            filtered_tokens.append(token)
            continue

        # Remove stopwords and single alphabets
        if token_lower in STOPWORDS or re.fullmatch(r"[a-z]", token_lower):
            continue

        filtered_tokens.append(token)

    # Reconstruct sentence
    stopword_sentence = " ".join(filtered_tokens)
    stopword_sentence = re.sub(r"\s+([,.;:!?])", r"\1", stopword_sentence)
    stopword_sentence = remove_duplicate_tokens(stopword_sentence)
    stopword_sentence = re.sub(r"\s+", " ", stopword_sentence).strip()

    row["stopword_sentence"] = stopword_sentence
    row["token_before"] = len(sentence.split())
    row["token_after"] = len(stopword_sentence.split())
    row["removed_tokens"] = row["token_before"] - row["token_after"]

# Save Output
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("\nStopword Removal Completed and Saved!")

### Stop Word Statistics
Let's review the reduction rate to see how many filler words were successfully eliminated.

In [ ]:
# =====================================================
# STOPWORD STATISTICS
# =====================================================
df_stop = pd.DataFrame(data)

before_mean = df_stop["token_before"].mean()
after_mean = df_stop["token_after"].mean()
reduction = before_mean - after_mean
reduction_rate = (reduction / before_mean) * 100 if before_mean > 0 else 0

print("=" * 60)
print("Stopword Statistics")
print("=" * 60)
print(f"Reviews               : {len(df_stop):,}")
print(f"Average Tokens Before : {before_mean:.2f}")
print(f"Average Tokens After  : {after_mean:.2f}")
print(f"Average Removed       : {reduction:.2f}")
print(f"Reduction Rate        : {reduction_rate:.2f}%")

# Preview
display(df_stop[["clean_sentence", "stopword_sentence"]].head(10))

In [ ]:
# =====================================================
# BEFORE vs AFTER STOP WORD REMOVAL
# =====================================================
review_df = df_stop[[
    "sentence_id",
    "clean_sentence",
    "stopword_sentence",
    "token_before",
    "token_after"
]]

display(review_df)

## 7. Aspect Extraction
In this module, we categorize the reviews into specific business aspects (e.g., Quality, Effectiveness, Delivery). We use a two-stage approach:
1. **Explicit Aspect Detection:** Looking for direct labels like `Quality:`.
2. **Keyword Matching:** Scanning the sentence for related terminology (e.g., "fast", "arrived" -> Delivery).

In [ ]:
# =====================================================
# ASPECT DICTIONARIES
# =====================================================
EXPLICIT_ASPECT = {
    "quality:": "quality",
    "effectiveness:": "effectiveness",
    "taste:": "taste",
    "delivery:": "delivery",
    "packaging:": "packaging",
    "price:": "price",
    "expiry:": "expiry",
    "seller:": "seller"
}

ASPECT_PRIORITY = [
    "quality", "effectiveness", "taste", "delivery",
    "packaging", "price", "seller", "expiry"
]

ASPECT_KEYWORDS = {
    "quality": {"quality", "genuine", "original", "authentic", "premium", "counterfeit", "fake", "fresh", "freshness", "defect", "defective"},
    "effectiveness": {"effective", "effectiveness", "energy", "boost", "stamina", "performance", "endurance", "recovery", "fuel", "hydrate", "hydration", "training", "running", "cycling", "ride", "workout", "exercise", "marathon", "helps", "help", "run", "cycle", "bike", "gym"},
    "taste": {"taste", "tastes", "flavour", "flavours", "flavor", "flavors", "sweet", "salty", "bitter", "sour", "delicious", "yummy", "awesome", "orange", "lemon", "cola", "berry", "apple", "grape"},
    "delivery": {"delivery", "deliver", "delivered", "arrived", "arrival", "shipping", "ship", "courier", "parcel", "postage", "fast", "slow", "late", "quick", "delay", "delayed"},
    "packaging": {"packaging", "package", "packed", "wrap", "wrapped", "bubble", "bubblewrap", "box", "sealed", "seal", "wrapper", "wrapping", "carton", "damage", "damaged", "well", "secure", "securely", "leak", "leaking", "broken", "crushed", "dent", "intact"},
    "price": {"price", "worth", "value", "cheap", "expensive", "promotion", "promo", "discount", "affordable", "pricing", "cost", "money", "rm", "ringgit"},
    "seller": {"seller", "service", "response", "respond", "reply", "replied", "support", "communicates", "communication", "accommodating", "responsive", "friendly", "polite", "trustworthy"},
    "expiry": {"expiry", "expire", "expired", "expiration", "exp"}
}

In [ ]:
# =====================================================
# ASPECT EXTRACTION LOGIC
# =====================================================
INPUT_FILE = OUTPUT_FOLDER / "09_stopwords_removed.json"
OUTPUT_ASPECT_FILE = OUTPUT_FOLDER / "10_aspects_extracted.json"
aspect_counter = Counter()

for row in data:
    sentence = str(row.get("stopword_sentence", "")).lower()
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", sentence)
    token_set = set(tokens)

    aspects, explicit_aspects = [], set()
    aspect_source, aspect_match_keyword = {}, {}

    # Stage 1: Explicit Aspect
    for prefix, aspect in EXPLICIT_ASPECT.items():
        if prefix in sentence:
            explicit_aspects.add(aspect)
            aspects.append(aspect)
            aspect_source[aspect] = "explicit"
            aspect_match_keyword[aspect] = [prefix]

    # Stage 2: Keyword Matching
    for aspect in ASPECT_PRIORITY:
        if aspect in explicit_aspects:
            continue

        hits = [kw for kw in ASPECT_KEYWORDS[aspect] if kw in token_set]

        if hits:
            aspects.append(aspect)
            aspect_source[aspect] = "keyword"
            aspect_match_keyword[aspect] = sorted(hits)

    # Remove duplicates and determine Primary Aspect
    aspects = list(dict.fromkeys(aspects))
    primary = next((asp for asp in ASPECT_PRIORITY if asp in aspects), None)

    # Save to row
    row.update({
        "aspect_keywords": aspects,
        "primary_aspect": primary,
        "aspect_count": len(aspects),
        "aspect_source": aspect_source,
        "aspect_match_keyword": aspect_match_keyword
    })

    aspect_counter.update(aspects)

# Save JSON
with open(OUTPUT_ASPECT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("\nAspect Extraction Completed and Saved!")

# Print Stats
aspect_df = pd.DataFrame(sorted(aspect_counter.items(), key=lambda x: x[1], reverse=True), columns=["Aspect", "Count"])
reviews_with_aspect = sum(row["aspect_count"] > 0 for row in data)
coverage = (reviews_with_aspect / len(data)) * 100

display(aspect_df)
print(f"\nTotal Reviews        : {len(data):,}")
print(f"Reviews with Aspect  : {reviews_with_aspect:,}")
print(f"Coverage             : {coverage:.2f}%")

In [ ]:
# =====================================================
# ASPECT REVIEW
# =====================================================
review = pd.DataFrame(data)

print("=" * 60)
print("Random Aspect Review")
print("=" * 60)

display(review[["sentence_id", "stopword_sentence", "aspect_keywords",
            "primary_aspect", "aspect_count", "aspect_source", "aspect_match_keyword"]].sample(30, random_state=42))

## 8. Lemmatization
Lemmatization converts words to their base dictionary form (e.g., "running" -> "run", "better" -> "good"). This reduces the vocabulary size and helps the model recognize that different word forms share the same meaning.

We use NLTK with Part-of-Speech (POS) tagging for accurate lemmatization, and we apply **Domain Correction Rules** to prevent the lemmatizer from incorrectly altering domain-specific sports and logistics terms (e.g., ensuring "energy gels" becomes "energy gel").

In [ ]:
# =====================================================
# INSTALL & IMPORT LIBRARIES
# =====================================================
import nltk
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# Download required NLTK resources
resources = [
    "punkt", "punkt_tab", "wordnet", "omw-1.4",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"
]

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Error downloading {resource}: {e}")

print("=" * 60)
print("NLTK Resources Ready")
print("=" * 60)

# Load Aspect Dataset from previous step
INPUT_FILE = OUTPUT_FOLDER / "10_aspects_extracted.json"
OUTPUT_LEMMA_FILE = OUTPUT_FOLDER / "11_lemmatized.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data):,} reviews for Lemmatization.")

In [ ]:
# =====================================================
# STEP 1 & 2: PRE-NORMALIZATION & TOKENIZATION
# =====================================================
TOKEN_PATTERN = re.compile(
    r"\d+(?:st|nd|rd|th)|\d+/\d+|\d+%|[A-Za-z]+(?:'[A-Za-z]+)?|[.,!?;:()/&+-]"
)

def normalize_sentence(sentence):
    sentence = str(sentence)
    sentence = re.sub(r"(\d+)\s*/\s*(\d+)", r"\1/\2", sentence) # Score
    sentence = re.sub(r"(\d+)\s*%", r"\1%", sentence)           # Percentage
    sentence = re.sub(r"(\d+)\s+(st|nd|rd|th)\b", r"\1\2", sentence) # Ordinal
    sentence = re.sub(r"\s*:\s*", ": ", sentence)               # Attribute colon
    sentence = re.sub(r"\s+", " ", sentence)                    # Multiple spaces
    return sentence.strip()

def tokenize_sentence(sentence):
    """Convert sentence into ordered tokens."""
    sentence = normalize_sentence(sentence)
    if sentence == "": return []
    return TOKEN_PATTERN.findall(sentence)

def get_wordnet_pos(pos_tag):
    """Convert Penn Treebank POS tag to WordNet POS tag."""
    if pos_tag.startswith("J"): return wordnet.ADJ
    if pos_tag.startswith("V"): return wordnet.VERB
    if pos_tag.startswith("N"): return wordnet.NOUN
    if pos_tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def pos_tag_sentence(sentence):
    """Tokenize a sentence and assign Penn Treebank POS tags."""
    tokens = tokenize_sentence(sentence)
    if not tokens: return []
    return pos_tag(tokens)

In [ ]:
# =====================================================
# STEP 3 & 4: CORE LEMMATIZATION & DOMAIN CORRECTION
# =====================================================
DOMAIN_CORRECTION = {
    ("goods","good"): "goods", ("items","item"): "items",
    ("packaged","package"): "packaged", ("wrapped","wrap"): "wrapped",
    ("boxed","box"): "boxed", ("sealed","seal"): "sealed",
    ("packaging","package"): "packaging", ("shipping","ship"): "shipping",
    ("sports","sport"): "sports", ("times","time"): "times",
    ("boss","bos"): "boss", ("tired","tire"): "tired",
    ("energised","energise"): "energised", ("energized","energize"): "energized"
}

def lemmatize_tokens(tagged_tokens):
    """Convert POS-tagged tokens into WordNet lemmas."""
    lemma_tokens = []
    for token, pos in tagged_tokens:
        lower_token = token.lower()

        # Keep punctuation, score, percentage, ordinals
        if (re.fullmatch(r"[.,!?;:()/&+-]", lower_token) or
            re.fullmatch(r"\d+/\d+", lower_token) or
            re.fullmatch(r"\d+%", lower_token) or
            re.fullmatch(r"\d+(?:st|nd|rd|th)", lower_token)):
            lemma_tokens.append({"original": token, "lemma": token, "pos": pos})
            continue

        wn_pos = get_wordnet_pos(pos)
        lemma = lemmatizer.lemmatize(lower_token, pos=wn_pos)
        lemma_tokens.append({"original": token, "lemma": lemma, "pos": pos})

    return lemma_tokens

def apply_domain_correction(lemma_tokens):
    """Restore domain-specific words incorrectly lemmatized by WordNet."""
    corrected_tokens = []
    for token in lemma_tokens:
        original = token["original"].lower()
        lemma = token["lemma"]
        key = (original, lemma)

        if key in DOMAIN_CORRECTION:
            token["lemma"] = DOMAIN_CORRECTION[key]
        corrected_tokens.append(token)
    return corrected_tokens

In [ ]:
# =====================================================
# STEP 5 & 6: RULE-BASED SINGULARIZER & REBUILDING
# =====================================================
DOMAIN_PRODUCT_NOUNS = {"gel", "bar", "drink", "powder", "capsule", "tablet", "chew", "shake"}
DOMAIN_ACTIVITY_NOUNS = {"run", "ride", "walk", "hike", "cycle", "workout", "marathon", "race", "training"}

def singularize(word):
    """Convert regular English plural nouns into singular form using morphological rules."""
    if len(word) <= 2: return word
    if word in {"us", "is", "this", "yes", "his"}: return word

    if word.endswith("ies"): return word[:-3] + "y"
    if re.search(r"(xes|ses|zes|ches|shes)$", word): return word[:-2]
    if word.endswith("s") and not word.endswith("ss"): return word[:-1]
    return word

def normalize_phrases(tokens):
    """Normalize domain-specific plural nouns."""
    normalized = [token.copy() for token in tokens]
    for token in normalized:
        lemma = token["lemma"]
        singular = singularize(lemma)
        if singular in DOMAIN_PRODUCT_NOUNS or singular in DOMAIN_ACTIVITY_NOUNS:
            token["lemma"] = singular
    return normalized

def build_sentence(tokens):
    """Convert processed tokens back to sentence."""
    sentence = " ".join(token["lemma"] for token in tokens)
    sentence = re.sub(r"\s+([.,!?;:])", r"\1", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

# Master Pipeline Function
def lemmatize_sentence_pipeline(sentence):
    tagged_tokens = pos_tag_sentence(sentence)
    lemma_tokens = lemmatize_tokens(tagged_tokens)
    lemma_tokens = apply_domain_correction(lemma_tokens)
    lemma_tokens = normalize_phrases(lemma_tokens)
    return build_sentence(lemma_tokens)

In [ ]:
# =====================================================
# EXECUTE & SAVE LEMMATIZATION
# =====================================================
modified_count = 0

for row in data:
    original = str(row.get("stopword_sentence", ""))
    lemma = lemmatize_sentence_pipeline(original)
    row["lemma_sentence"] = lemma

    if original != lemma:
        modified_count += 1

with open(OUTPUT_LEMMA_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("=" * 60)
print("LEMMATIZATION COMPLETED")
print("=" * 60)
print(f"Reviews            : {len(data):,}")
print(f"Modified Reviews   : {modified_count:,}")
print(f"Modification Rate  : {(modified_count/len(data))*100:.2f}%")

In [ ]:
# =====================================================
# REVIEW MODIFIED SENTENCES
# =====================================================
df_stop = pd.DataFrame(data)
changed = df_stop[df_stop["stopword_sentence"]!=df_stop["lemma_sentence"]]

print("=" * 60)
print(f"Modified Reviews : {len(changed):,}")
print("=" * 60)

display(changed[["sentence_id", "stopword_sentence", "lemma_sentence"]]
    .sample(min(30, len(changed)), random_state=42))

## 9. Sentiment Labeling & Data Deduplication
Here, we use a pre-trained **RoBERTa** model (`cardiffnlp/twitter-roberta-base-sentiment-latest`) to score the textual sentiment of our translated sentences.

To create our **Final Sentiment Score**, we blend the text sentiment (80% weight) with the previously extracted emoji sentiment (20% weight).

Finally, we apply deduplication strategies (both exact string matching and fuzzy matching via `rapidfuzz`) to ensure our dataset is perfectly balanced and clean before training our own models.

In [ ]:
# =====================================================
# INSTALL & IMPORT
# =====================================================
!pip install transformers torch accelerate rapidfuzz -q
from transformers import pipeline
import pandas as pd
from rapidfuzz import fuzz

# Load Lemmatized Dataset
df = pd.read_json(OUTPUT_LEMMA_FILE)

# Initialize RoBERTa Sentiment Pipeline
# GPU is recommended for this step!
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 # Set to -1 if running on CPU
)

In [ ]:
# =====================================================
# TEXT SENTIMENT PREDICTION & BLENDING
# =====================================================
def predict_sentiment(text):
    if text is None or str(text).strip() == "":
        return "neutral", 0

    result = classifier(str(text))[0]
    label = result["label"].lower()
    conf = result["score"]

    if label == "positive": score = conf
    elif label == "negative": score = -conf
    else: score = 0

    return label, score

# Apply RoBERTa prediction
print("Running RoBERTa sentiment analysis... (This might take a few minutes)")
df[["text_sentiment", "text_score"]] = df["translated_sentence"].apply(
    lambda x: pd.Series(predict_sentiment(x))
)

# Blend Text (80%) + Emoji (20%)
df["final_score"] = (0.8 * df["text_score"]) + (0.2 * df["emoji_score"])

def score_to_label(score):
    if score >= 0.20: return "positive"
    elif score <= -0.20: return "negative"
    else: return "neutral"

df["final_sentiment"] = df["final_score"].apply(score_to_label)

# Save intermediate labeled data
OUTPUT_LABELED_CSV = OUTPUT_FOLDER / "12_sentiment_labeled.csv"
OUTPUT_LABELED_JSON = OUTPUT_FOLDER / "12_sentiment_labeled.json"
df.to_csv(OUTPUT_LABELED_CSV, index=False)
df.to_json(OUTPUT_LABELED_JSON, orient='records', indent=2)

print("\nSentiment Labeling Complete!")
display(df[["translated_sentence", "text_score", "emoji_score", "final_score", "final_sentiment"]].head(5))

In [ ]:
# =====================================================
# FILTERING & DEDUPLICATION (CLEANUP FOR TRAINING)
# =====================================================
print(f"Starting Row Count: {len(df)}")

# 1. Filter out pure "Delivery" and "Seller" reviews
# These aspects often skew sentiment models because they don't reflect the product itself.
df = df[~df["aspect_keywords"].apply(
    lambda x: isinstance(x, list) and ("delivery" in x or "seller" in x)
)].reset_index(drop=True)

print(f"Rows after removing Delivery/Seller aspects: {len(df)}")

# 2. Exact Deduplication (Lowercase & Stripped)
df["translated_sentence"] = df["translated_sentence"].str.lower().str.strip()
df = df.drop_duplicates(subset=["translated_sentence"]).reset_index(drop=True)
print(f"Rows after exact deduplication: {len(df)}")

# 3. Fuzzy Deduplication (Remove 95%+ similarity)
keep = []
for sentence in df["translated_sentence"]:
    duplicated = False
    for exist in keep:
        if fuzz.ratio(sentence, exist) >= 95:
            duplicated = True
            break
    if not duplicated:
        keep.append(sentence)

df = df[df["translated_sentence"].isin(keep)].reset_index(drop=True)
print(f"Final Row Count after Fuzzy Deduplication: {len(df)}")

# Save final ready-for-training dataset
OUTPUT_FINAL_CSV = OUTPUT_FOLDER / "13_remove_repeat.csv"
OUTPUT_FINAL_JSON = OUTPUT_FOLDER / "13_remove_repeat.json"

df.to_csv(OUTPUT_FINAL_CSV, index=False)
df.to_json(OUTPUT_FINAL_JSON, orient='records', indent=2)

print("\nFinal Label Distribution:")
print(df["final_sentiment"].value_counts(dropna=False))

## 9.5 Data Balancing (Undersampling)
Our dataset currently has an imbalanced distribution of sentiment labels. To prevent our models from becoming biased toward the majority class, we will perform **Random Undersampling**.

This technique finds the class with the smallest number of examples and randomly samples the exact same number of examples from the other larger classes, resulting in a perfectly balanced dataset.

In [ ]:
# =====================================================
# RANDOM UNDERSAMPLING
# =====================================================
import pandas as pd

# Load the cleaned, deduplicated dataset
INPUT_FILE = OUTPUT_FOLDER / "13_remove_repeat.json"
df_ml = pd.read_json(INPUT_FILE)

print("Original Label Distribution:")
print(df_ml["final_sentiment"].value_counts())
print("-" * 40)

# 1. Find the size of the smallest class
min_class_size = df_ml["final_sentiment"].value_counts().min()
print(f"Target size for balancing: {min_class_size} rows per class")

# 2. Undersample all classes to match the smallest class
df_balanced = (
    df_ml.groupby("final_sentiment")
    .apply(lambda x: x.sample(n=min_class_size, random_state=42))
    .reset_index(drop=True)
)

# 3. Shuffle the balanced dataset so the classes are randomly distributed
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("-" * 40)
print("Balanced Label Distribution:")
print(df_balanced["final_sentiment"].value_counts())

# Overwrite df_ml so the downstream training modules use the balanced data
df_ml = df_balanced

# (Optional) Save the balanced dataset if you want to inspect it later
OUTPUT_BALANCED_JSON = OUTPUT_FOLDER / "14_final_training_data.json"
df_ml.to_json(OUTPUT_BALANCED_JSON, orient='records', indent=2)
print(f"\nBalanced dataset saved to {OUTPUT_BALANCED_JSON}")